# CSCI 347 Project 2
## Problem 1 - Dataset Familiarization

Dataset chosen: **Twitch Social Networks (ENGB split)**

Dataset link: https://snap.stanford.edu/data/twitch-social-networks.html

#### 1) Why did you choose this dataset, and why is it interesting to you? (Make sure you include a link to the dataset that you chose).

I thought it was interesting because Twitch is an application that I am familiar with, however know very little about. I know that people use it to stream pretty much anything and can make money off of it, but I wanted to understand more about how people are connected thorugh the application.  

#### 2) Did you preprocess the data? If so, how did you preprocess it? Clearly explain the process. e.g., Did you take the largest connected component? Was it manageable, if not, did you sample it? How did you take the sample of the graph? What did you do after sampling? Did you do any other preprocessing? etc. 



#### 3) Explain what type of data is represented in this dataset.

The data set represents Twitch streamers who stream in specific languages. The nodes represents the users themselves while the links represent the mutual friendships between each streamer. 

#### 4a)  What characteristic do you expect the nodes with high centrality to possess? E.g., what type of non-graph characteristic would you expect the nodes with high centrality have? Note that this answer depends on the dataset that you choose.

I expect the nodes with high centrality to be streamers who are the most "famous" or have the most views. Since more people want to play with them, they would have a lot of friends on all of their games. 

#### 4b) Do you think this graph dataset would exhibit power law? State your reasons.

Yes, I believe that most of the streamers of Twitch would be very lowly viewed streamers, so their friend count would be slim on every game - those people added being closer or person friends. 

#### 4c) Do you expect this graph to have small-world property? State your reasons.

Yes, I am somewhat thinking it follows the principals of Facebook where it also has small world properties and exhibits the power law as well. 

## Problem 2 - Function Implementations

The following cells implement all 12 required functions for a simple, undirected, unweighted graph using an edge-list input.


In [6]:
from collections import defaultdict, deque
import numpy as np

def load_edge_list_txt(path):
    """Load whitespace-separated edge list file into list[tuple[int,int]]."""
    edges = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            u, v = line.split()[:2]
            edges.append((int(u), int(v)))
    return edges

def _build_adj(edges):
    adj = defaultdict(set)
    for u, v in edges:
        if u == v:
            continue
        adj[u].add(v)
        adj[v].add(u)
    return adj

def _bfs_distances(adj, source):
    if source not in adj:
        return {source: 0}
    dist = {source: 0}
    q = deque([source])
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                q.append(v)
    return dist

# 1) Number of nodes
def number_of_nodes(edges):
    nodes = set()
    for u, v in edges:
        nodes.add(u)
        nodes.add(v)
    return len(nodes)

# 2) Degree of a vertex
def degree_of_vertex(edges, vertex):
    adj = _build_adj(edges)
    return len(adj.get(vertex, set()))

# 3) Adjacency matrix
def adjacency_matrix(edges, return_nodes=False):
    nodes = sorted({x for e in edges for x in e})
    idx = {node: i for i, node in enumerate(nodes)}
    n = len(nodes)
    A = np.zeros((n, n), dtype=int)
    for u, v in edges:
        i, j = idx[u], idx[v]
        A[i, j] = 1
        A[j, i] = 1
    if return_nodes:
        return A, nodes
    return A

# 4) Degree distribution as list of counts
def degree_distribution(edges):
    adj = _build_adj(edges)
    if not adj:
        return []
    degs = [len(adj[u]) for u in adj]
    max_deg = max(degs)
    dist = [0] * (max_deg + 1)
    for d in degs:
        dist[d] += 1
    return dist

# 5) P( degree >= k )
def prob_degree_geq(deg_dist, k):
    total = sum(deg_dist)
    if total == 0:
        return 0.0
    if k <= 0:
        return 1.0
    if k >= len(deg_dist):
        return 0.0
    return sum(deg_dist[k:]) / total

# 6) Eccentricity of a vertex
def eccentricity_of_vertex(vertex, edges):
    adj = _build_adj(edges)
    nodes = set(adj.keys())
    if vertex not in nodes:
        return float("inf")
    dist = _bfs_distances(adj, vertex)
    if len(dist) != len(nodes):
        return float("inf")
    return max(dist.values())

# 7) Diameter of graph
def diameter_of_graph(edges):
    adj = _build_adj(edges)
    nodes = list(adj.keys())
    if not nodes:
        return 0
    diam = 0
    for u in nodes:
        dist = _bfs_distances(adj, u)
        if len(dist) != len(nodes):
            return float("inf")
        ecc = max(dist.values())
        if ecc > diam:
            diam = ecc
    return diam

# 8) Radius of graph
def radius_of_graph(edges):
    adj = _build_adj(edges)
    nodes = list(adj.keys())
    if not nodes:
        return 0
    radius = float("inf")
    for u in nodes:
        dist = _bfs_distances(adj, u)
        if len(dist) != len(nodes):
            return float("inf")
        ecc = max(dist.values())
        if ecc < radius:
            radius = ecc
    return radius

# 9) Clustering coefficient of a vertex
def clustering_coefficient_of_vertex(vertex, edges):
    adj = _build_adj(edges)
    nbrs = list(adj.get(vertex, []))
    k = len(nbrs)
    if k < 2:
        return 0.0
    links = 0
    for i in range(k):
        u = nbrs[i]
        for j in range(i + 1, k):
            v = nbrs[j]
            if v in adj[u]:
                links += 1
    return (2 * links) / (k * (k - 1))

# 10) Betweenness centrality of a vertex (Brandes, unweighted graph)
def betweenness_centrality_of_vertex(vertex, edges):
    adj = _build_adj(edges)
    nodes = list(adj.keys())
    if vertex not in adj:
        return 0.0

    cb_v = 0.0
    for s in nodes:
        stack = []
        pred = {w: [] for w in nodes}
        sigma = {w: 0 for w in nodes}
        dist = {w: -1 for w in nodes}
        sigma[s] = 1
        dist[s] = 0
        q = deque([s])

        while q:
            v = q.popleft()
            stack.append(v)
            for w in adj[v]:
                if dist[w] < 0:
                    q.append(w)
                    dist[w] = dist[v] + 1
                if dist[w] == dist[v] + 1:
                    sigma[w] += sigma[v]
                    pred[w].append(v)

        delta = {w: 0.0 for w in nodes}
        while stack:
            w = stack.pop()
            for v in pred[w]:
                if sigma[w] != 0:
                    delta[v] += (sigma[v] / sigma[w]) * (1.0 + delta[w])
            if w != s and w == vertex:
                cb_v += delta[w]

    return cb_v / 2.0

# 11) Closeness centrality of a vertex
def closeness_centrality_of_vertex(vertex, edges):
    adj = _build_adj(edges)
    nodes = set(adj.keys())
    if vertex not in nodes:
        return 0.0
    dist = _bfs_distances(adj, vertex)
    if len(dist) != len(nodes):
        return 0.0
    total = sum(dist.values())
    if total == 0:
        return 0.0
    return (len(nodes) - 1) / total

# 12) Eigenvector centrality with power iteration and L2 normalization
def eigenvector_centrality_power_iteration(A, max_iter=200, tol=1e-9):
    A = np.asarray(A, dtype=float)
    n = A.shape[0]
    x = np.ones(n, dtype=float)
    x /= np.linalg.norm(x, ord=2)

    for _ in range(max_iter):
        x_new = A @ x
        norm = np.linalg.norm(x_new, ord=2)
        if norm == 0:
            return x_new
        x_new /= norm
        if np.linalg.norm(x_new - x, ord=2) < tol:
            x = x_new
            break
        x = x_new
    return x


### Problem 2 Tests on `facebook_combined`

This cell runs each function at least once on SNAP `facebook_combined.txt`.


In [8]:
import os

candidate_paths = [
    "data/facebook_combined.txt",
    "Data/facebook_combined.txt"
]
facebook_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if facebook_path is None:
    raise FileNotFoundError("Could not find facebook_combined.txt")

print("Using dataset:", facebook_path)
fb_edges = load_edge_list_txt(facebook_path)
test_vertex = 0

print("#1 Number of nodes:", number_of_nodes(fb_edges))
print("#2 Degree of vertex", test_vertex, ":", degree_of_vertex(fb_edges, test_vertex))

A_fb, fb_nodes = adjacency_matrix(fb_edges, return_nodes=True)
print("#3 Adjacency matrix shape:", A_fb.shape)

deg_dist = degree_distribution(fb_edges)
print("#4 Degree distribution length:", len(deg_dist), "(max degree + 1)")

k = 10
print(f"#5 P(degree >= {k}):", prob_degree_geq(deg_dist, k))

print("#6 Eccentricity of vertex", test_vertex, ":", eccentricity_of_vertex(test_vertex, fb_edges))
print("#7 Graph diameter:", diameter_of_graph(fb_edges))
print("#8 Graph radius:", radius_of_graph(fb_edges))

print("#9 Clustering coefficient of vertex", test_vertex, ":", clustering_coefficient_of_vertex(test_vertex, fb_edges))
print("#10 Betweenness centrality of vertex", test_vertex, ":", betweenness_centrality_of_vertex(test_vertex, fb_edges))
print("#11 Closeness centrality of vertex", test_vertex, ":", closeness_centrality_of_vertex(test_vertex, fb_edges))

eig = eigenvector_centrality_power_iteration(A_fb, max_iter=200)
top_idx = np.argsort(eig)[-10:][::-1]
top_nodes = [(fb_nodes[i], float(eig[i])) for i in top_idx]
print("#12 Top 10 nodes by eigenvector centrality (node, score):")
for item in top_nodes:
    print(item)


Using dataset: data/facebook_combined.txt
#1 Number of nodes: 4039
#2 Degree of vertex 0 : 347
#3 Adjacency matrix shape: (4039, 4039)
#4 Degree distribution length: 1046 (max degree + 1)
#5 P(degree >= 10): 0.7858380787323594
#6 Eccentricity of vertex 0 : 6
#7 Graph diameter: 8
#8 Graph radius: 4
#9 Clustering coefficient of vertex 0 : 0.04196165314587463
#10 Betweenness centrality of vertex 0 : 1192496.1130793944
#11 Closeness centrality of vertex 0 : 0.35334266713335666
#12 Top 10 nodes by eigenvector centrality (node, score):
(1912, 0.09540586441536048)
(2266, 0.08698334097289433)
(2206, 0.08605252459444215)
(2233, 0.08517347290415829)
(2464, 0.08427890447770185)
(2142, 0.08419319953071394)
(2218, 0.08415586188670601)
(2078, 0.0841362965424963)
(2123, 0.08367154192081247)
(1993, 0.08353255553300797)


#### Generated Data runtime 1m 45.5s
##### Using dataset: data/facebook_combined.txt
1. Number of nodes: 4039
2. Degree of vertex 0 : 347
3. Adjacency matrix shape: (4039, 4039)
4. Degree distribution length: 1046 (max degree + 1)
5. P(degree >= 10): 0.7858380787323594
6. Eccentricity of vertex 0 : 6
7. Graph diameter: 8
8. Graph radius: 4
9. Clustering coefficient of vertex 0 : 0.04196165314587463
10. Betweenness centrality of vertex 0 : 1192496.1130793944
11. Closeness centrality of vertex 0 : 0.35334266713335666
12. Top 10 nodes by eigenvector centrality (node, score):
* (1912, 0.09540586441536048)
* (2266, 0.08698334097289433)
* (2206, 0.08605252459444215)
* (2233, 0.08517347290415829)
* (2464, 0.08427890447770185)
* (2142, 0.08419319953071394)
* (2218, 0.08415586188670601)
* (2078, 0.0841362965424963)
* (2123, 0.08367154192081247)
* (1993, 0.08353255553300797)

## Problem 3

In [ ]:
import networkx as nx
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load edge list
edges = pd.read_csv("musae_ENGB_edges.csv")

# Convert to list of tuples
edge_list = list(zip(edges['from'], edges['to']))

# Build graph
G = nx.Graph()
G.add_edges_from(edge_list)

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

# Largest connected component
largest_cc = max(nx.connected_components(G), key=len)
G = G.subgraph(largest_cc).copy()

print("Nodes in LCC:", G.number_of_nodes())
print("Edges in LCC:", G.number_of_edges())

Problem 3.1

In [ ]:
# sample 500 nodes for visualization
sample_nodes = list(G.nodes())[:500]
H = G.subgraph(sample_nodes)

plt.figure(figsize=(8,8))
nx.draw(H,
        node_size=10,
        edge_color="gray",
        with_labels=False)

plt.title("Sample Visualization of Twitch Social Network")
plt.show()

Problem 3.2

In [ ]:
closeness = nx.closeness_centrality(G)

top_closeness = sorted(closeness.items(),
                       key=lambda x: x[1],
                       reverse=True)[:10]

top_closeness

pd.DataFrame(top_closeness,
             columns=["Node","Closeness Centrality"])

Problem 3.3

In [ ]:
betweenness = nx.betweenness_centrality(G)

top_betweenness = sorted(betweenness.items(),
                         key=lambda x: x[1],
                         reverse=True)[:10]

pd.DataFrame(top_betweenness,
             columns=["Node","Betweenness Centrality"])

Problem 3.4

In [ ]:
clustering = nx.clustering(G)

top_clustering = sorted(clustering.items(),
                        key=lambda x: x[1],
                        reverse=True)[:10]

pd.DataFrame(top_clustering,
             columns=["Node","Clustering Coefficient"])

Problem 3.5

In [ ]:
eigenvector = nx.eigenvector_centrality(G, max_iter=1000)

top_eigenvector = sorted(eigenvector.items(),
                         key=lambda x: x[1],
                         reverse=True)[:10]

pd.DataFrame(top_eigenvector,
             columns=["Node","Eigenvector Centrality"])

Problem 3.6

In [ ]:
pagerank = nx.pagerank(G)

top_pagerank = sorted(pagerank.items(),
                      key=lambda x: x[1],
                      reverse=True)[:10]

pd.DataFrame(top_pagerank,
             columns=["Node","PageRank"])

Problem 3.7

Problem 3.8

In [ ]:
degrees = [d for n, d in G.degree()]

degree_counts = np.bincount(degrees)
k = np.arange(len(degree_counts))

# remove zero values
mask = degree_counts > 0
k = k[mask]
freq = degree_counts[mask] / sum(degree_counts)

# log transform
log_k = np.log(k)
log_freq = np.log(freq)

# least squares fit
coef = np.polyfit(log_k, log_freq, 1)
fit_line = np.poly1d(coef)

plt.figure(figsize=(7,5))
plt.scatter(log_k, log_freq)
plt.plot(log_k, fit_line(log_k))

plt.xlabel("log(k)")
plt.ylabel("log(f(k))")
plt.title("Log-Log Degree Distribution")

plt.show()

print("Slope:", coef[0])